# Benchmark: Inclined Plane

In [1]:
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import sys
import cartopy.crs as ccrs
import subprocess
import os
import pandas as pd
import xarray as xr
import rasterio as rio
from rasterio.enums import Resampling
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor

#### In this experiment, an isothermal, isoviscous flow of a Newtonian fluid is erupted from a point source onto a sloping plane.

This is based on a benchmark previously defined for lava flow models by Cordonnier et al. [2015](https://appliedvolc.biomedcentral.com/articles/10.1186/s13617-017-0061-x#ref-CR14); BM2). It is a scenario that was explored analytically and experimentally by Lister ([1992](https://appliedvolc.biomedcentral.com/articles/10.1186/s13617-017-0061-x#ref-CR48)), and is well understood, capturing the dominant behavior of a lava flow as a viscous gravity current. 

We encourage changing the volume betweens runs to compare the physical or stochastic reactions at various scales. It is also recommended to change the angle to measure various internal friction effects.

In [2]:
angle = 15

volume = 1.0804e-3

#### The following cells should not be edited, unless you have extensive knowledge of the model.

The following cell sets up common parameters. Following this, the next 4 cells format and generate input files for each respective model, at parameters considered reasonable for this benchmark.

In [19]:
os.chdir('/home/jovyan/Lava Benchmark')

easting = 0
northing = 0
scale = volume/1.0804e-3

f = open("DEMs_Lava_Benchmark/slope.asc","r+")
header = f.readlines()
header[2] = f"""xllcorner {-0.3515*scale}\n"""
header[3] = f"""yllcorner {-0.4515*scale}\n"""
header[4] = f"""cellsize {0.003*scale}\n"""
f.seek(0)
f.writelines(header)
f.truncate()
f.close()

r = rxr.open_rasterio("DEMs_Lava_Benchmark/slope.asc")
height = r.rio.resolution()[0]*501*np.sin(np.deg2rad(angle))
slope = np.linspace(height,0,501)
stacked = np.tile(slope,(301,1))
r[0,:,:] = stacked
r.rio.to_raster("DEMs_Lava_Benchmark/Benchmark_Inclined.asc")
coordinates = np.array([int(easting),int(northing)])

dem = "DEMs_Lava_Benchmark/Benchmark_Inclined.asc"
raster = rxr.open_rasterio(dem)
res = raster.rio.resolution()[0]
raster = raster.sel({"band": 1})
xll, yll = float(raster.x.min()), float(raster.y.min())

### Molasses: Additional Notes

Molasses input parameters are stored in `custom_molasses.conf` in the `Molasses` folder. The most impactful change would be that of changing the minimum and maximum residual, as the cellular automata structure will cause a much larger or smaller area to be impacted. As a stochastic model, no physical parameters are used.

In [6]:
os.chdir('/home/jovyan/Lava Benchmark/Molasses')
f = open("custom_molasses.conf","r+")
conf = f.readlines()
conf[2] = f"""MIN_RESIDUAL = {1e-6*scale}\n"""
conf[3] = f"""MAX_RESIDUAL ={1e-2*scale}\n"""
conf[4] = f"""MIN_TOTAL_VOLUME = {str(volume)}\n"""
conf[5] = f"""MAX_TOTAL_VOLUME = {str(volume)}\n"""
conf[6] = f"""MIN_PULSE_VOLUME = {float(volume)/100}\n"""
conf[7] = f"""MAX_PULSE_VOLUME = {float(volume)/100}\n"""
conf[10] = f"""DEM_FILE = ../{dem}\n"""
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

f=open("events.in","w")
f.write(f"""{easting},{northing}""")
f.close()

subprocess.run("molasses custom_molasses.conf",shell=True, stderr=subprocess.DEVNULL)

victor.convert_molasses("molasses_inclined")

Seeding random number generator: 1743000444


               MOLASSES is a lava flow simulator.

Config file: custom_molasses.conf
Reading in Parameters...
             PARENTS = 1                    [assigned]
    ELEVATION_UNCERT = 0                    [assigned]
        MIN_RESIDUAL = 1e-06                [assigned]
        MAX_RESIDUAL = 0.01                 [assigned]
    MIN_TOTAL_VOLUME = 0.0010804            [assigned]
    MAX_TOTAL_VOLUME = 0.0010804            [assigned]
    MIN_PULSE_VOLUME = 1.0804e-05           [assigned]
    MAX_PULSE_VOLUME = 1.0804e-05           [assigned]
                RUNS = 1                    [assigned]
      ASCII_FLOW_MAP = 1                    [assigned]
            DEM_FILE = ../DEMs_Lava_Benchmark/Benchmark_Inclined.asc [assigned]
[assigned]VENTS_FILE = events.in            
Nothing missing.

DEM Information [Arc/Info ASCII Grid]:
  File:              ../DEMs_Lava_Benchmark/Benchmark_Inclined.asc
  Lower Left Origin: (-0.351500,-0.451500)
  

### MrLavaLoba/Flowy: Additional Notes

MrLavaLoba/Flowy (a drop in C++ replacement) input parameters are stored in `input.toml` in the `flowy` folder. The most impactful changes would be those such as the thickening parameter, lobe exponent, or max slope probability. Other parameters not mentioned will also affect the creation and placement of lobes. As a stochastic model, no physical parameters are specified by the user.

In [22]:
os.chdir('/home/jovyan/Lava Benchmark/flowy')
g=open("input.toml","r+")
inp = g.readlines()
inp[1] = f"""run_name ='Inclined'\n"""
inp[6] = f"""source = '../{dem}'\n"""
inp[21] = """vent_flag = 0\n"""
inp[24] = f"""x_vent = [{int(easting)}]\n"""
inp[25] = f"""y_vent = [{int(northing)}]\n"""
inp[63] = f"""total_volume = {volume}\n"""
g.seek(0)
g.writelines(inp)
g.truncate()
g.close()

subprocess.run("./flowy input.toml",shell=True)

Using input file: input.toml
Output directory path set to: ./output
run_name = Inclined_004
total_time = 00h 00m 00.058s
Total number of processed lobes = 5305
n_lobes/ms = 91
Used RNG seed: -144141263


CompletedProcess(args='./flowy input.toml', returncode=0)

In [27]:
os.chdir('/home/jovyan/Lava Benchmark/IMEX_LavaFlow')
radius = 2e-3/scale
h=open("IMEX_LavaFlow.inp","r+")
inp = h.readlines()
inp[1] = 'RUN_NAME="Inclined           ",\n'
inp[4] = f""" T_END=  600,\n"""
inp[5] = f""" DT_OUTPUT=  120,\n"""
inp[13] = f""" X0= {xll}D0     ,\n"""
inp[14] = f""" Y0= {yll}D0      ,\n"""
inp[15] = " COMP_CELLS_X=  500,\n"
inp[16] = " COMP_CELLS_Y=  300,\n"
inp[17] = f""" CELL_SIZE = {res}D0 ,\n"""
inp[103] = f"""  X_SOURCE=  {easting}.0D0 ,\n"""
inp[104] = f"""  Y_SOURCE=  {northing}.0D0 ,\n"""
inp[105] = f""" R_SOURCE=  {res*2}     ,\n"""
inp[106] = f""" VEL_SOURCE= {float(volume)/(radius**2*np.pi*600)} \n"""
h.seek(0)
h.writelines(inp)
h.truncate()
h.close()

subprocess.run(f"""ln -s ../{dem} topography_dem.asc""",shell=True)
p = subprocess.Popen(['./IMEX_LavaFlow'], stdin=subprocess.PIPE, shell=True)
p.communicate(input=b'\n')

ln: failed to create symbolic link 'topography_dem.asc': File exists


 ---------------------
 IMEX_LavaFlow 1.0
 ---------------------
 OpenMP program
 Number of threads used           4
 Run name: Inclined                                
 ----- 2D SIMULATION -----
 Ambient density =    1.1763298740177413       (kg/m3)
 CARRIER PHASE: liquid
 Carrier phase kinematic viscosity:  -1.0000000000000000     
 radiative_term_coeff =    2.2680000000000002E-008
 convective_term_coeff =    35.000000000000000     
 Searching for DEM file
 Reading DEM file
 ncols         501
 nrows         301
 Percent Complete: 100.00% 
 IOSTAT=          -1
 thickness_levels          -1
 dyn_pres_levels          -1
 Searching for probes coords
 Source area =   1.1310119910864160E-004  Error =   3.4161542276534007E-005
 Implicit equations =           3

 ******** START COMPUTATION *********

 WRITING Inclined_0000.p_2d                      
 WRITING Inclined_0000.asc                       
 WRITING Inclined_T_0000.asc                     
 WRITING Inclined_T_surf_0000.asc           

KeyboardInterrupt: 

In [34]:
os.chdir('/home/jovyan/Lava Benchmark/lava2d')

victor.create_geotiff(
    data=stacked,
    output_path='Benchmark_inc.tiff',
    x_min=1000,  # easting
    y_min=1000,  # northing
    pixel_width=.03,  # 10 meter pixels
    pixel_height=-.03,  # negative because image coordinates go down
    crs='EPSG:32610',  # UTM Zone 10N example
    nodata_value=-9999
)
r = rxr.open_rasterio("Benchmark_inc.tiff",masked=True)
lonlat = r.rio.reproject("EPSG:4326")
xmin,ymin,xmax,ymax = lonlat.rio.bounds()
center = float(lonlat[:,150,115].x), float(lonlat[:,150,115].y)

k=open("./input.py","r+")
inp = k.readlines()
inp[4] = '    path_to_dem_file    = ("./Benchmark_inc.tiff"),\n'
inp[5] = f"""    Lon_SRC             = {center[0]}, # source longitude\n"""
inp[6] = f"""    Lat_SRC             = {center[1]},    # source latitude\n"""
inp[7] = f"""    Lon_LowerLeft       = {xmin}, # bounding box: lower-left longitude\n"""
inp[8] = f"""    Lat_LowerLeft       = {ymin}, # bounding box: lower-left latitude\n"""
inp[9] = f"""    Lon_UpperRight      = {xmax}, # bounding box: upper-right longitude\n"""
inp[10] = f"""    Lat_UpperRight      = {ymax},   # bounding box: upper-right latitude\n"""
inp[67] = f"""    max_time_hr = {600/3600},\n"""

k.seek(0)
k.writelines(inp)
k.truncate()
k.close()

k=open("./example_vents/vent_01.txt","r+")
inp = k.readlines()
inp[1] = f'''0	{-.03*scale}	{-.03*scale}	{.03*scale}	{.03*scale}	{.01*scale}	{7.7e-4*scale}\n'''

k.seek(0)
k.writelines(inp)
k.truncate()
k.close()

subprocess.run("python input.py", shell=True)

GeoTIFF created successfully with rasterio at Benchmark_inc.tiff
| --- Reading DEM Dataset --- |
| --- Building Modelling Subset --- |
| --- Building Coordinate Transform --- |
| --- Preparing DEM --- |
| --- DEM Preparation Complete --- |
| --- Initializing Model Domain --- |
| --- Beginning Model Evolution --- |
|  Step (sub): 10 (1)  |  dx: 0.11999999999999979 m  |  Grid: 17 x 17  |  dt (type): 0.016 s (par)  |  t: 0.00 hr |
|  Step (sub): 20 (1)  |  dx: 0.11999999999999979 m  |  Grid: 17 x 17  |  dt (type): 0.005 s (par)  |  t: 0.00 hr |
|  Step (sub): 30 (1)  |  dx: 0.11999999999999979 m  |  Grid: 17 x 17  |  dt (type): 0.005 s (par)  |  t: 0.00 hr |
|  Step (sub): 40 (1)  |  dx: 0.11999999999999979 m  |  Grid: 17 x 17  |  dt (type): 0.005 s (par)  |  t: 0.00 hr |
|  Step (sub): 50 (1)  |  dx: 0.11999999999999979 m  |  Grid: 33 x 33  |  dt (type): 0.005 s (par)  |  t: 0.00 hr |
|  Step (sub): 60 (1)  |  dx: 0.11999999999999979 m  |  Grid: 33 x 33  |  dt (type): 0.004 s (par)  |  t

CompletedProcess(args='python input.py', returncode=0)

In [ ]:
data = xr.open_dataset("lava2d/outputs/out.nc",group="DATA/PHYSICS",decode_coords="all")
data = data.where(data!=0, np.nan)

coord = xr.open_dataset("lava2d/outputs/out.nc",group="DATA", decode_coords="all")
lat = coord.lat[:,0]
lon = coord.lon[0,:]
data['rows'] = lat.values
data['cols'] = lon.values
data = data.rename({"rows": "y", "cols": "x"})
repeated_arr_x = np.repeat(data.y.values[0], len(data.x))
repeated_arr_y = np.repeat(data.x.values[0], len(data.y))
data.lava_thickness_total.rio.to_raster("lava2d/inclined_output.tif")

#### The final two cells concern visualization.

The first provides a reference to the final timestamp of each simulation, though all outputs are avilable in their models' respective folder. The second and last cell plots each model side by side. We encourage users to edit this to their personal preference.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/")

flowy_outputs = glob.glob(f'flowy/output/Inclined*thickness_masked*.asc')
flowy_out = flowy_outputs[-1]

r = rxr.open_rasterio("IMEX_LavaFlow/Inclined_max.asc",masked=True)
r2 = rxr.open_rasterio("Molasses/molasses_inclined.asc",masked=True)
r3 = rxr.open_rasterio(flowy_out,masked=True)
r4 = rxr.open_rasterio("lava2d/inclined_output.tif",masked=True)
r_min, r_max = r.min(), r.max()
r2_min, r2_max = r2.min(), r2.max()
r3_min, r3_max = r3.min(), r3.max()
r4_min, r4_max = r4.min()/1000, r4.max()/1000

maximum = np.max((r_max,r2_max,r3_max,r4_max))
idx_max = np.argmax((r_max,r2_max,r3_max,r4_max))

In [ ]:
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2, figsize = (12,9))


thickness0 = victor.plot_flow(dem, "Molasses/molasses_inclined.asc", axes=ax0,zoom=False,title='Molasses',minimum=1e-6,scale=maximum)
thickness1 = victor.plot_flow(dem, flowy_out, axes=ax1, zoom=False,title='MrLavaLoba/Flowy',minimum=1e-6, scale=maximum)
thickness2 = victor.plot_flow(dem, "IMEX_LavaFlow/Inclined_max.asc", axes=ax2, zoom=False,title='IMEX_LavaFlow', minimum=1e-6, scale=maximum)
thickness3 = victor.plot_flow("lava2d/Benchmark_Inclined.tiff", "lava2d/inclined_output.tif", axes=ax3 ,title='Lava2D', minimum=1e-10, scale=maximum*100)

xticks = np.linspace(-.3515, 1.15, 5)
yticks = np.linspace(-.4515, .4515, 5)

cbar_ax = fig.add_axes([.25, 0, 0.5, 0.05])
thickness = [thickness0, thickness1, thickness2, thickness3]
thickness = thickness[idx_max]
fig.colorbar(thickness, cax=cbar_ax, label="Flow Thickness (m)", orientation="horizontal")